# Warm-start

**Este notebook só RODA e SALVA as buscas** em `results/curvas/<método>.npz` (curvas, configurações avaliadas e desenhos iniciais de cada alvo e semente). O que já está em `results/curvas/` é lido em vez de recalculado (use `FORCE_SEARCH = True` para refazer, aprox. 45 min).

**Foi usado protocolo leave-one-dataset-out.** Avaliar uma configuração é ler a célula de
d_t na matriz de perdas. d_t sai da matriz de perdas usada para montar o warm-start e da normalização das meta-características. Todo o meta-conhecimento vem dos outros 119. **T = 100** avaliações por busca, **n_init = 20** pontos iniciais e **5 sementes**,
iguais para todos os métodos:

1. **Configuração padrão**: fixa, sem busca;
2. **Random Search**: permutação aleatória das K configurações, sem reposição;
3. **SMBO-Cold**: partida fria com 20 configurações aleatórias;
4. **SMBO-Similarity (k)**: warm-start por datasets similares. Os 20 pontos iniciais são as melhores configurações dos *k* datasets mais próximos de d_t no espaço das meta-features, em rodízio (a melhor de cada vizinho, depois a 2ª melhor de cada um, ...); k ∈ {1, 3, 5}. Com k = 1 são as 20 melhores do vizinho mais próximo. Com k ≥ 20 seria a melhor de cada um dos 20 vizinhos, como em Feurer et al. (2014).
5. **SMBO-Independent**: warm-start independente dos dados. As 20 configurações de melhor rank médio nos 119 datasets de treino, sem olhar para d_t.

Os métodos 3–5 usam o mesmo SMBO e só diferem nos 20 pontos iniciais:

- Modelo surrogate é o GP com kernel Matérn 3/2 e ARD (um comprimento de escala por dimensão da codificação), com os hiperparâmetros do GP reajustados a cada iteração. As configurações são categórizadas em one-hot e as numéricas são normalizadas por min-max.
- Aquisição: Expected Improvement, calculado sobre todas as configurações do grid ainda não avaliadas; a próxima a ser avaliada é a de maior EI.
Empates são frequentes, porque muitas configurações têm a mesma acurácia de validação. Nesses casos foi feito sorteio para ordenar as configurações de cada vizinho (SMBO-Similarity) e ao escolher o máximo do EI.
- Parada antecipada: quando a busca encontra a melhor configuração do grid, ela é interrompida.

In [1]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "src" / "config.py").exists():
    ROOT = Path(r"D:\Projetos\DisciplinaMtL\entrega3")
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as cfg, search, analysis as an
from src.metatask_data import load_meta_task

an.setup_matplotlib()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
cfg.make_dirs()

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(cfg.FIGURES_DIR / f"{name}.{ext}", bbox_inches="tight", facecolor="white")

def save_table(df, name, index=True):
    df.to_csv(cfg.TABLES_DIR / f"{name}.csv", index=index)

FORCE_SEARCH = False     
N_JOBS = -2            

meta = load_meta_task()
ctx = search.build_context(meta)
KEYS = search.all_method_keys()
print(f"{len(ctx['names'])} alvos | K={meta.n_cfg} | T={ctx['T']} | n_init={ctx['n_init']} | sementes={ctx['n_seeds']} | "
      f"{len(KEYS)} métodos: {[search.method_label(k) for k in KEYS]}")
print("Configuração padrão:", meta.configs.iloc[ctx['default_idx']].to_dict())

120 alvos | K=3456 | T=100 | n_init=20 | sementes=5 | 7 métodos: ['Configuração padrão', 'Random Search', 'SMBO-Cold', 'SMBO-Independent', 'SMBO-Similarity (k=1)', 'SMBO-Similarity (k=3)', 'SMBO-Similarity (k=5)']
Configuração padrão: {'ACTIVATION': 'relu', 'ARCHITECTURE': 'SQU', 'DROPOUT': 0.0, 'LAYERS': 1, 'NEURONS': 32, 'NORMALIZATION': 0, 'OPTIMZERS': 'adam'}


In [2]:
RES = {}
for key in KEYS:
    cached = None if FORCE_SEARCH else search.load_method(key, ctx)
    RES[key] = cached if cached is not None else search.run_method(key, ctx, n_jobs=N_JOBS, force=FORCE_SEARCH)
    print(f"{'cache ' if cached is not None else 'rodado'} | {search.method_label(key):18s} | "
          f"tempo {RES[key]['info']['wall_s']:6.0f} s | serial {RES[key]['info']['serial_s']:6.0f} s")

cache  | Configuração padrão | tempo      0 s | serial      0 s
cache  | Random Search      | tempo      0 s | serial      0 s
cache  | SMBO-Cold          | tempo    485 s | serial   5184 s
cache  | SMBO-Independent   | tempo    489 s | serial   5198 s
cache  | SMBO-Similarity (k=1) | tempo    515 s | serial   5512 s
cache  | SMBO-Similarity (k=3) | tempo    477 s | serial   5055 s
cache  | SMBO-Similarity (k=5) | tempo    422 s | serial   4365 s


Perda normalizada média (sobre alvos e sementes) logo após os pontos iniciais, no orçamento intermediário e em T.

In [3]:
#Só pra testar se tá funcionando rs

quick = pd.DataFrame({search.method_label(k): {f"t={t}": r["curves"][:, :, t - 1].mean() for t in cfg.CHECKPOINTS}
                      for k, r in RES.items()}).T
quick["AUC"] = [r["curves"].mean() for r in RES.values()]
quick["tempo de CPU acumulado (s)"] = [r["info"]["serial_s"] for r in RES.values()]
quick["tempo decorrido (s)"] = [r["info"]["wall_s"] for r in RES.values()]
save_table(quick, "buscas_conferencia")
display(quick.style.format("{:.4f}", subset=[c for c in quick.columns if not c.startswith("tempo")])
        .format("{:.0f}", subset=["tempo de CPU acumulado (s)", "tempo decorrido (s)"]))
print(f"Custo total das buscas simuladas: {quick['tempo de CPU acumulado (s)'].sum() / 3600:.1f} h de CPU "
      f"({quick['tempo decorrido (s)'].sum() / 3600:.1f} h); "
      f"{len(KEYS) * len(ctx['names']) * ctx['n_seeds']:,} buscas.")

,t=20,t=50,t=100,AUC,tempo de CPU acumulado (s),tempo decorrido (s)
Configuração padrão,0.1700,0.1700,0.1700,0.1700,0,0
Random Search,0.0834,0.0606,0.0467,0.0722,0,0
SMBO-Cold,0.0842,0.0520,0.0375,0.0652,5184,485
SMBO-Independent,0.0755,0.0526,0.0377,0.0604,5198,489
SMBO-Similarity (k=1),0.0733,0.0453,0.0341,0.0549,5512,515
SMBO-Similarity (k=3),0.0635,0.0452,0.0338,0.0531,5055,477
SMBO-Similarity (k=5),0.0645,0.0438,0.0326,0.0525,4365,422


Custo total das buscas simuladas: 7.0 h de CPU (0.7 h); 4,200 buscas.
